# Databricks backend clustering job
This notebook uses a configurable backend URL to check service health, verify ML endpoints, and trigger clustering. Set `backend_base_url` as a Databricks widget or environment variable when configuring the job.

In [ ]:
import os
import json
from datetime import datetime, timezone

def _widget(name: str, default: str) -> str:
    try:
        value = dbutils.widgets.get(name)  # type: ignore[name-defined]
        return value or default
    except Exception:
        return os.getenv(name.upper(), default)

BACKEND_BASE_URL = _widget("backend_base_url", "https://scout-backend-um0t.onrender.com").rstrip("/")
REQUEST_TIMEOUT_SECONDS = int(_widget("request_timeout_seconds", "120"))
CLUSTER_LIMIT = int(_widget("cluster_limit", "500"))
N_CLUSTERS = int(_widget("n_clusters", "8"))

if not BACKEND_BASE_URL:
    raise ValueError(
        "Set backend_base_url to a reachable backend URL. Databricks cannot use 127.0.0.1 unless the backend is running in the same cluster environment."
    )

if BACKEND_BASE_URL.startswith("http://127.0.0.1") or BACKEND_BASE_URL.startswith("http://localhost") or BACKEND_BASE_URL.startswith("https://127.0.0.1") or BACKEND_BASE_URL.startswith("https://localhost"):
    raise ValueError(
        "backend_base_url points to localhost. Use a public or cluster-reachable URL from Databricks instead.",
    )
        {
            "cell_type": "code",
            "id": "#VSC-8c1d39de",
            "metadata": {
                "language": "python"
            },
            "source": [
                "import requests",
                "from sentence_transformers import SentenceTransformer",
                "from sklearn.cluster import KMeans",
                "import json",
                "",
                "session = requests.Session()",
                "base_url = BACKEND_BASE_URL.rstrip(\"/\")",
                "",
                "def request_json(method: str, path: str, **kwargs):",
                "    url = f\"{base_url}{path}\"",
                "    response = session.request(method, url, timeout=REQUEST_TIMEOUT_SECONDS, **kwargs)",
                "    response.raise_for_status()",
                "    try:",
                "        return response.json()",
                "    except ValueError:",
                "        return {\"text\": response.text}",
                "",
                "# Health checks",
                "try:",
                "    health = request_json(\"get\", \"/health\")",
                "    ml_health = request_json(\"get\", \"/ml/health\")",
                "    ml_status = request_json(\"get\", \"/ml/status\")",
                "",
                "    print(\"Backend health:\")",
                "    print(json.dumps(health, indent=2, default=str))",
                "    print(\"\\nML health:\")",
                "    print(json.dumps(ml_health, indent=2, default=str))",
                "    print(\"\\nML status:\")",
                "    print(json.dumps(ml_status, indent=2, default=str))",
                "",
                "    # Fetch normalized events from backend and run heavy ML locally in Databricks",
                "    events = request_json(\"get\", \"/ml/events/export\", params={\"limit\": CLUSTER_LIMIT})",
                "    texts = [e.get(\"summary\", \"\") or \"\" for e in events]",
                "",
                "    if not texts:",
                "        print(\"No events returned for clustering\")",
                "    else:",
                "        print(\"Generating embeddings with sentence-transformers model...\")",
                "        model = SentenceTransformer(\"sentence-transformers/all-mpnet-base-v2\")",
                "        embeddings = model.encode(texts, show_progress_bar=True)",
                "",
                "        print(\"Running KMeans clustering in Databricks...\")",
                "        kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42)",
                "        labels = kmeans.fit_predict(embeddings)",
                "",
                "        payload = [",
                "            {\"event_id\": events[i][\"event_id\"], \"cluster_id\": int(labels[i])} for i in range(len(events))",
                "        ]",
                "",
                "        print(\"Saving cluster assignments back to backend...\")",
                "        save_result = request_json(\"post\", \"/ml/clusters/save\", json=payload)",
                "        print(\"Save result:\")",
                "        print(json.dumps(save_result, indent=2, default=str))",
                "",
                "        summary = {",
                "            \"backend_base_url\": BACKEND_BASE_URL,",
                "            \"health_status\": health.get(\"status\"),",
                "            \"classifier_loaded\": ml_health.get(\"classifier_loaded\"),",
                "            \"classifier_model\": ml_health.get(\"classifier_model\"),",
                "            \"clustered_count\": len(payload),",
                "            \"saved\": save_result.get(\"saved\"),",
                "        }",
                "",
                "        print(\"\\nRun summary:\")",
                "        print(json.dumps(summary, indent=2, default=str))",
                "except requests.RequestException as exc:",
                "    raise RuntimeError(",
                "        f\"Failed to reach backend at {BACKEND_BASE_URL}. Set backend_base_url to a publicly reachable URL from Databricks. Original error: {exc}\",",
                "    ) from exc",
            ]
        }
    ) from exc